# Nemotron Submit Packager — tinker_plus_5e5_full_010

Lightweight notebook that picks the best merged adapter from the CV kernel output and packages it as `submission.zip` ready for **Submit to Competition**.

**No GPU required.** Attach `aleixlopez/nemotron-local-cv` as a kernel source.


In [ ]:
# Package a chosen merged adapter from the CV kernel output into submission.zip.
# No GPU required — this notebook runs in seconds.
import os, zipfile
from pathlib import Path

# ── choose which adapter to package ──────────────────────────────────────────
# Must match a name in the CV kernel's merged_adapter_manifest.csv.
ADAPTER_NAME = "tinker_plus_5e5_full_010"
# ─────────────────────────────────────────────────────────────────────────────

# Locate the CV kernel output.  It's mounted as a kernel-source input so it
# lands under /kaggle/input/<kernel-owner>/<kernel-slug>/.
import glob
cv_candidates = glob.glob(
    f"/kaggle/input/**/merged_adapters/{ADAPTER_NAME}/adapter_config.json",
    recursive=True,
)
assert cv_candidates, (
    f"Could not find adapter_config.json for {ADAPTER_NAME} under /kaggle/input. "
    "Make sure the CV kernel is attached as a kernel source."
)
adapter_dir = Path(cv_candidates[0]).parent
print(f"adapter_dir: {adapter_dir}")

REQUIRED = ["adapter_config.json", "adapter_model.safetensors"]
missing = [f for f in REQUIRED if not (adapter_dir / f).is_file()]
assert not missing, f"Missing required files: {missing}"

out_zip = Path("/kaggle/working/submission.zip")
out_zip.parent.mkdir(parents=True, exist_ok=True)
if out_zip.exists():
    out_zip.unlink()

with zipfile.ZipFile(out_zip, "w", zipfile.ZIP_DEFLATED) as zf:
    for fname in REQUIRED:
        zf.write(adapter_dir / fname, arcname=fname)

# Verify
with zipfile.ZipFile(out_zip) as zf:
    names = sorted(zf.namelist())
assert names == sorted(REQUIRED), f"Unexpected zip contents: {names}"

size_gb = out_zip.stat().st_size / 1024**3
print(f"wrote {out_zip}  ({size_gb:.2f} GB)")
print(f"contents: {names}")
print("Ready → click Output > submission.zip > Submit to Competition")
